In [1]:
from pathlib import Path
from typing import Literal

import pandas as pd
from datasets import Dataset, load_dataset
from utils import upload_file
from huggingface_hub import HfApi
from sentence_transformers.util import mine_hard_negatives
from sentence_transformers import SentenceTransformer

In [2]:
repo_id = "Studeni/amazon-esci-data"
cache_dir = Path("../.cache")
api = HfApi()

## Queries

In [ ]:
queries = load_dataset(
    path="Studeni/amazon-esci-data",
    name="queries",
    split=["train", "test"],
    cache_dir=cache_dir,
)

In [ ]:
queries

In [ ]:
df_queries = pd.concat([queries[0].to_pandas(), queries[1].to_pandas()])
print(f"Number of rows in queries dataset: {len(df_queries):_}")

In [6]:
del queries

In [ ]:
df_queries.head()

In [ ]:
examples = """
## Examples starts here
Here are the cells in this Jupyter Notebook:
`CELL INDEX: 0
```python
examples = df_queries[df_queries["product_locale"] == "es"]["query"].unique().tolist()
with open("examples.txt", "w") as file:
    for example in examples:
        file.write(f"{example}\n")
```

In [ ]:
df_queries[df_queries["product_locale"] == "es"]["query"].unique().tolist()
with open("examples.txt", "w") as file:
    for example in examples:
        file.write(f"{example}\n")

## Utils

Desired Query Columns:
- "query_id", 
- "query",
- "query_locale"/"product_locale"
- "product_ids"

In [63]:
def process_queries(
    df_queries: pd.DataFrame,
    version: Literal["small", "large"],
    esci_label: Literal["E", "S", "C", "I"] = "E",
) -> pd.DataFrame:
    version_filter = "small_version" if version == "small" else "large_version"

    df = (
        df_queries[
            (df_queries["esci_label"] == esci_label) & (df_queries[version_filter] == 1)
        ]
        .groupby("query_id")
        .agg(
            {
                "query": "first",
                "product_locale": "first",  # All are the same, checked with: lambda x: x.unique().tolist(),
                "product_id": list,
                "split": "first",
            }
        )
        .reset_index()
    )
    df.rename(
        columns={"product_locale": "query_locale", "product_id": "pos_product_ids"},
        inplace=True,
    )
    return df

### Exact Query - Products | Small/Hard

In [ ]:
df = process_queries(df_queries, version="small", esci_label="E")
print(f"Number of rows in processed queries dataset: {len(df):_}")
df.head()

Upload to HF

In [47]:
upload_file(
    api=api,
    df=df,
    path_in_repo="retrieval/exact_queries_small.parquet",
    repo_id=repo_id,
    repo_type="dataset",
)

### Exact Query - Products | Small/Hard

In [ ]:
df = process_queries(df_queries, version="large", esci_label="E")
print(f"Number of rows in processed queries dataset: {len(df):_}")
df.head()

Upload to HF

In [68]:
upload_file(
    api=api,
    df=df,
    path_in_repo="retrieval/exact_queries_large.parquet",
    repo_id=repo_id,
    repo_type="dataset",
)

## HARD Negative Mining

In [ ]:
model_name = "intfloat/e5-base-v2"
model = SentenceTransformer(model_name_or_path=model_name, cache_folder=cache_dir)

In [ ]:
mine_hard_negatives()